In [1]:
import pandas as pd
import time
from datetime import datetime
import csv

In [2]:
# ---------------------------------------------------------
# 1. READ SOURCE CSV
# ---------------------------------------------------------

df = pd.read_csv("GlobalWeatherRepository.csv")

print("CSV file loaded successfully!")
print("Total source records:", len(df))

CSV file loaded successfully!
Total source records: 164499


In [3]:
# ---------------------------------------------------------
# 2. CHECK REQUIRED COLUMNS
# ---------------------------------------------------------

required_columns = [
    "location_name",
    "latitude",
    "longitude",
    "temperature_celsius",
    "humidity"
]

for column in required_columns:
    if column not in df.columns:
        raise ValueError("Column not found: " + column)

print("Required columns found!")

Required columns found!


In [4]:
# ---------------------------------------------------------
# 3. REMOVE EMPTY RECORDS
# ---------------------------------------------------------

df = df.dropna(subset=required_columns).reset_index(drop=True)

print("Valid source records:", len(df))

Valid source records: 164499


In [5]:
# ---------------------------------------------------------
# 4. CREATE CONTAINER IDs
# ---------------------------------------------------------

container_ids = {}

for number, location in enumerate(
    df["location_name"].unique(),
    start=1
):
    container_ids[location] = "CONT_{:04d}".format(number)

print("Container IDs created!")

Container IDs created!


In [6]:
# ---------------------------------------------------------
# 5. OUTPUT FILE
# ---------------------------------------------------------

output_file = "AtmoSync_Streaming_Data.csv"

columns = [
    "Container_ID",
    "Timestamp",
    "Temperature_C",
    "Humidity_Percent",
    "Latitude",
    "Longitude"
]


# Create a fresh output file
with open(
    output_file,
    "w",
    newline="",
    encoding="utf-8"
) as file:

    writer = csv.DictWriter(
        file,
        fieldnames=columns
    )

    writer.writeheader()


print("Output file created:", output_file)

Output file created: AtmoSync_Streaming_Data.csv


In [7]:
# ---------------------------------------------------------
# 6. GENERATE EXACTLY 100 RECORDS
# ---------------------------------------------------------

print()
print("==============================================")
print("ATMOSYNC STREAMING STARTED")
print("==============================================")
print("Generating exactly 100 records...")
print("One record every 2 seconds")
print()


record_number = 0

try:

    while record_number < 100:

        # Select source row
        row = df.iloc[record_number % len(df)]

        # Increase record number
        record_number = record_number + 1

        # Container ID
        container_id = container_ids[
            row["location_name"]
        ]

        # Current timestamp
        timestamp = datetime.now().strftime(
            "%Y-%m-%d %H:%M:%S"
        )

        # Create AtmoSync record
        record = {
            "Container_ID": container_id,
            "Timestamp": timestamp,
            "Temperature_C": row["temperature_celsius"],
            "Humidity_Percent": row["humidity"],
            "Latitude": row["latitude"],
            "Longitude": row["longitude"]
        }

        # Save record to CSV
        with open(
            output_file,
            "a",
            newline="",
            encoding="utf-8"
        ) as file:

            writer = csv.DictWriter(
                file,
                fieldnames=columns
            )

            writer.writerow(record)

        # Display record
        print(
            "Record {:03d} | Container: {} | "
            "Temperature: {} C | Humidity: {}% | "
            "Location: {}".format(
                record_number,
                container_id,
                row["temperature_celsius"],
                row["humidity"],
                row["location_name"]
            )
        )

        # Wait before next record
        if record_number < 100:
            time.sleep(2)


except KeyboardInterrupt:

    print()
    print("Streaming manually stopped.")


ATMOSYNC STREAMING STARTED
Generating exactly 100 records...
One record every 2 seconds

Record 001 | Container: CONT_0001 | Temperature: 26.6 C | Humidity: 24% | Location: Kabul
Record 002 | Container: CONT_0002 | Temperature: 19.0 C | Humidity: 94% | Location: Tirana
Record 003 | Container: CONT_0003 | Temperature: 23.0 C | Humidity: 29% | Location: Algiers
Record 004 | Container: CONT_0004 | Temperature: 6.3 C | Humidity: 61% | Location: Andorra La Vella
Record 005 | Container: CONT_0005 | Temperature: 26.0 C | Humidity: 89% | Location: Luanda
Record 006 | Container: CONT_0006 | Temperature: 26.0 C | Humidity: 84% | Location: Saint John's
Record 007 | Container: CONT_0007 | Temperature: 8.0 C | Humidity: 93% | Location: Buenos Aires
Record 008 | Container: CONT_0008 | Temperature: 19.0 C | Humidity: 40% | Location: Yerevan
Record 009 | Container: CONT_0009 | Temperature: 9.0 C | Humidity: 87% | Location: Canberra
Record 010 | Container: CONT_0010 | Temperature: 16.0 C | Humidity: 6

In [8]:
# ---------------------------------------------------------
# 7. FINAL MESSAGE
# ---------------------------------------------------------

print()
print("==============================================")
print("ATMOSYNC STREAMING COMPLETED")
print("==============================================")
print("Total records generated:", record_number)
print("Data saved in:", output_file)


ATMOSYNC STREAMING COMPLETED
Total records generated: 100
Data saved in: AtmoSync_Streaming_Data.csv


In [9]:
# ---------------------------------------------------------
# 8. VERIFY GENERATED DATA
# ---------------------------------------------------------

streaming_df = pd.read_csv("AtmoSync_Streaming_Data.csv")

print("AtmoSync streaming file loaded!")
print("Total records:", len(streaming_df))

print()
print("First 5 records:")
display(streaming_df.head())

print()
print("Last 5 records:")
display(streaming_df.tail())

print()
print("Columns:")
print(streaming_df.columns.tolist())

AtmoSync streaming file loaded!
Total records: 100

First 5 records:


,Container_ID,Timestamp,Temperature_C,Humidity_Percent,Latitude,Longitude
0,CONT_0001,2026-09-15 20:03:45,26.6,24,34.52,69.18
1,CONT_0002,2026-09-15 20:03:47,19.0,94,41.33,19.82
2,CONT_0003,2026-09-15 20:03:49,23.0,29,36.76,3.05
3,CONT_0004,2026-09-15 20:03:51,6.3,61,42.50,1.52
4,CONT_0005,2026-09-15 20:03:53,26.0,89,-8.84,13.23



Last 5 records:


,Container_ID,Timestamp,Temperature_C,Humidity_Percent,Latitude,Longitude
95,CONT_0096,2026-09-15 20:06:56,18.0,52,-29.32,27.48
96,CONT_0097,2026-09-15 20:06:58,26.0,94,6.31,-10.80
97,CONT_0098,2026-09-15 20:07:00,33.0,63,18.78,100.78
98,CONT_0099,2026-09-15 20:07:02,15.0,72,47.13,9.52
99,CONT_0100,2026-09-15 20:07:04,17.0,39,54.68,25.32



Columns:
['Container_ID', 'Timestamp', 'Temperature_C', 'Humidity_Percent', 'Latitude', 'Longitude']


In [10]:
# ---------------------------------------------------------
# 9. CHECK DATA QUALITY
# ---------------------------------------------------------

print("Missing values:")
print(streaming_df.isnull().sum())

print()
print("Duplicate records:")
print(streaming_df.duplicated().sum())

Missing values:
Container_ID        0
Timestamp           0
Temperature_C       0
Humidity_Percent    0
Latitude            0
Longitude           0
dtype: int64

Duplicate records:
0
